In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sakshamboran/echo-72k/Y_train_full.npy
/kaggle/input/datasets/sakshamboran/echo-72k/scaling.json
/kaggle/input/datasets/sakshamboran/echo-72k/Y_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/Y_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_train_full.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_train_full.npy


In [2]:
!curl -I https://huggingface.co --max-time 10

HTTP/2 200 
content-type: text/html; charset=utf-8
content-length: 174586
date: Wed, 15 Jul 2026 09:49:23 GMT
etag: W/"2a9fa-JnBoedRPnytsrtgV3kfXtkeFeAA"
x-powered-by: huggingface-moon
x-request-id: Root=1-6a5757a3-5ac5549937fabc97453971c3
ratelimit: "pages";r=97;t=129
ratelimit-policy: "fixed window";"pages";q=100;w=300
cross-origin-opener-policy: same-origin
referrer-policy: strict-origin-when-cross-origin
x-frame-options: DENY
x-cache: Hit from cloudfront
via: 1.1 713041d22f2aa4c7f5ed7c21c1b4fa88.cloudfront.net (CloudFront)
x-amz-cf-pop: ORD58-P13
alt-svc: h3=":443"; ma=86400
x-amz-cf-id: H4iRkAsdNkBYyynCUK9Sy6VsSZpzo_fRpc_2Zw1kif9dLZpHYfxXUg==
age: 58



In [3]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/sakshamboran/echo-72k/Y_train_full.npy
/kaggle/input/datasets/sakshamboran/echo-72k/scaling.json
/kaggle/input/datasets/sakshamboran/echo-72k/Y_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/Y_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_test.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_train_full.npy
/kaggle/input/datasets/sakshamboran/echo-72k/X_val.npy
/kaggle/input/datasets/sakshamboran/echo-72k/T_train_full.npy


In [4]:
!git clone https://github.com/Jwoo5/fairseq-signals.git /content/fairseq-signals 2>&1 | tail -3
%cd /content/fairseq-signals
!pip install --editable ./ 2>&1 | tail -15

Cloning into '/content/fairseq-signals'...
/content/fairseq-signals
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.4 MB/s eta 0:00:00
  Building editable for fairseq_signals (pyproject.toml): started
  Building editable for fairseq_signals (pyproject.toml): finished with status 'done'
  Created wheel for fairseq_signals: filename=fairseq_signals-1.0.0a0+f8f0ff1-0.editable-cp312-cp312-linux_x86_64.whl size=6635 sha256=6c9b04b16965dda5836c7f89e2a974bc9274d51ec97e0bc275f1b0c422d44191
  Stored in directory: /tmp/pip-ephem-wheel-cache-9xsd7z2e/wheels/9a/e6/fd/7847bdc7c0122549e05ab9a85afaf7479add63bf3bee83f1ae
Successfully built fairseq_signals


In [5]:
!pip install huggingface_hub -q
from huggingface_hub import hf_hub_download
import os
os.makedirs('/kaggle/working/ecgfm_ckpt', exist_ok=True)
ckpt_path = hf_hub_download(
    repo_id="wanglab/ecg-fm",
    filename="mimic_iv_ecg_physionet_pretrained.pt",
    local_dir="/kaggle/working/ecgfm_ckpt",
)
print("Checkpoint:", ckpt_path)

mimic_iv_ecg_physionet_pretrained.pt:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Checkpoint: /kaggle/working/ecgfm_ckpt/mimic_iv_ecg_physionet_pretrained.pt


In [6]:
import sys
sys.path.insert(0, '/content/fairseq-signals')
import torch
from fairseq_signals.models import build_model_from_checkpoint

model = build_model_from_checkpoint(checkpoint_path='/kaggle/working/ecgfm_ckpt/mimic_iv_ecg_physionet_pretrained.pt')
model.eval()
print("Model built:", type(model))
print("Params:", sum(p.numel() for p in model.parameters())/1e6, "M")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

Model built: <class 'fairseq_signals.models.wav2vec2.wav2vec2_cmsc.Wav2Vec2CMSCModel'>
Params: 90.883072 M
Device: cuda


In [7]:
import numpy as np, json
from torch.utils.data import Dataset, DataLoader
from scipy.signal import resample_poly

DATA = '/kaggle/input/datasets/sakshamboran/echo-72k'

with open(f'{DATA}/scaling.json') as f:
    SCALING = json.load(f)
print("Tabular columns:", SCALING['columns'])

class EchoNextDataset12L(Dataset):
    def __init__(self, x_path, t_path, y_path):
        self.X = np.load(x_path, mmap_mode='r')
        self.T = np.load(t_path)
        self.Y = np.load(y_path)
        assert len(self.X) == len(self.T) == len(self.Y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        ecg = np.array(self.X[idx, 0])
        ecg = resample_poly(ecg, 2, 1, axis=0)
        ecg = ecg.T.astype(np.float32)
        t = self.T[idx].astype(np.float32)
        y = self.Y[idx].astype(np.float32)
        return torch.tensor(ecg), torch.tensor(t), torch.tensor(y)

train_ds = EchoNextDataset12L(f'{DATA}/X_train_full.npy', f'{DATA}/T_train_full.npy', f'{DATA}/Y_train_full.npy')
val_ds   = EchoNextDataset12L(f'{DATA}/X_val.npy', f'{DATA}/T_val.npy', f'{DATA}/Y_val.npy')
test_ds  = EchoNextDataset12L(f'{DATA}/X_test.npy', f'{DATA}/T_test.npy', f'{DATA}/Y_test.npy')

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
ecg0, t0, y0 = train_ds[0]
print("ecg:", ecg0.shape, "tabular:", t0.shape, "labels:", y0)

Tabular columns: ['ventricular_rate', 'atrial_rate', 'pr_interval', 'qrs_duration', 'qt_corrected', 'age_at_ecg', 'sex']
train=72475 val=4626 test=5442
ecg: torch.Size([12, 5000]) tabular: torch.Size([7]) labels: tensor([0., 0., 0., 0.])


In [8]:
import torch.nn as nn

class MultiTargetHFModel(nn.Module):
    def __init__(self, ecgfm_backbone, tabular_dim=7, hidden=256, n_targets=4, freeze_backbone_frac=0.7):
        super().__init__()
        self.backbone = ecgfm_backbone
        params = list(self.backbone.parameters())
        n_freeze = int(len(params) * freeze_backbone_frac)
        for p in params[:n_freeze]:
            p.requires_grad = False
        print(f"Froze {n_freeze}/{len(params)} backbone parameter tensors")
        embed_dim = 768
        self.tabular_mlp = nn.Sequential(nn.Linear(tabular_dim, 32), nn.ReLU(), nn.Dropout(0.1))
        fused_dim = embed_dim + 32
        self.head_trunk = nn.Sequential(nn.Linear(fused_dim, hidden), nn.ReLU(), nn.Dropout(0.2))
        self.heads = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(n_targets)])

    def forward(self, ecg, tabular):
        out = self.backbone(source=ecg)
        pooled = out['features'].mean(dim=1)
        tab = self.tabular_mlp(tabular)
        fused = torch.cat([pooled, tab], dim=1)
        h = self.head_trunk(fused)
        return torch.cat([head(h) for head in self.heads], dim=1)

net = MultiTargetHFModel(model).to(device)
print(f"Trainable: {sum(p.numel() for p in net.parameters() if p.requires_grad)/1e6:.1f}M / {sum(p.numel() for p in net.parameters())/1e6:.1f}M")

Froze 150/215 backbone parameter tensors
Trainable: 25.5M / 91.1M


In [9]:
import torch.nn.functional as F

def masked_bce_loss(logits, targets, pos_weights):
    losses = []
    for i in range(targets.shape[1]):
        col_t, col_l = targets[:, i], logits[:, i]
        mask = ~torch.isnan(col_t)
        if mask.sum() == 0: continue
        losses.append(F.binary_cross_entropy_with_logits(col_l[mask], col_t[mask], pos_weight=pos_weights[i]))
    return torch.stack(losses).mean()

pos_rates = torch.tensor([0.1789, 0.1324, 0.2438, 0.5237])
pos_weights = ((1 - pos_rates) / pos_rates).to(device)
print("pos_weights:", pos_weights)

pos_weights: tensor([4.5897, 6.5529, 3.1017, 0.9095], device='cuda:0')


In [10]:
from sklearn.metrics import roc_auc_score
import numpy as np, time

BATCH_SIZE = 32
EPOCHS = 8
PATIENCE = 3

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

backbone_params = [p for p in net.backbone.parameters() if p.requires_grad]
new_params = list(net.tabular_mlp.parameters()) + list(net.head_trunk.parameters()) + list(net.heads.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 2e-6},
    {'params': new_params, 'lr': 1e-4},
])

best_val_auroc = -1
patience_ctr = 0

for epoch in range(EPOCHS):
    net.train()
    train_losses = []
    t0 = time.time()
    for i, (ecg, t, y) in enumerate(train_loader):
        ecg, t, y = ecg.to(device), t.to(device), y.to(device)
        optimizer.zero_grad()
        loss = masked_bce_loss(net(ecg, t), y, pos_weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
        if i % 200 == 0:
            print(f"  epoch {epoch}, batch {i}/{len(train_loader)}, loss={loss.item():.4f}, elapsed={time.time()-t0:.0f}s")

    net.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for ecg, t, y in val_loader:
            logits = net(ecg.to(device), t.to(device))
            all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
    all_logits, all_y = np.concatenate(all_logits), np.concatenate(all_y)
    m = ~np.isnan(all_y[:, 0])
    val_auroc = roc_auc_score(all_y[m, 0], all_logits[m, 0])
    print(f"Epoch {epoch}: train_loss={np.mean(train_losses):.4f}  val_HFrEF_AUROC={val_auroc:.4f}")

    torch.save(net.state_dict(), f'/kaggle/working/ecgfm_full72k_epoch{epoch}.pt')
    print(f"  -> checkpoint saved: ecgfm_full72k_epoch{epoch}.pt")

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        patience_ctr = 0
        torch.save(net.state_dict(), '/kaggle/working/ecgfm_full72k_BEST.pt')
        print("  -> also saved as BEST")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val HFrEF AUROC: {best_val_auroc:.4f}")

  epoch 0, batch 0/2265, loss=1.6568, elapsed=4s
  epoch 0, batch 200/2265, loss=0.8903, elapsed=367s
  epoch 0, batch 400/2265, loss=0.8829, elapsed=733s
  epoch 0, batch 600/2265, loss=0.7837, elapsed=1099s
  epoch 0, batch 800/2265, loss=0.8862, elapsed=1465s
  epoch 0, batch 1000/2265, loss=0.8105, elapsed=1831s
  epoch 0, batch 1200/2265, loss=0.9157, elapsed=2198s
  epoch 0, batch 1400/2265, loss=0.8966, elapsed=2563s
  epoch 0, batch 1600/2265, loss=0.8976, elapsed=2929s
  epoch 0, batch 1800/2265, loss=0.5304, elapsed=3295s
  epoch 0, batch 2000/2265, loss=0.7341, elapsed=3662s
  epoch 0, batch 2200/2265, loss=0.7925, elapsed=4028s
Epoch 0: train_loss=0.8296  val_HFrEF_AUROC=0.9051
  -> checkpoint saved: ecgfm_full72k_epoch0.pt
  -> also saved as BEST
  epoch 1, batch 0/2265, loss=0.8494, elapsed=2s
  epoch 1, batch 200/2265, loss=0.6062, elapsed=368s
  epoch 1, batch 400/2265, loss=0.5483, elapsed=735s
  epoch 1, batch 600/2265, loss=0.6419, elapsed=1101s
  epoch 1, batch 800/